# Lecture 2 — Class Exercise
# Bar Charts: World Happiness Report 2023


In [2]:
import pandas as pd
import numpy as np

# World Happiness Report 2023
df = pd.read_csv('../data/world_happiness_2023.csv')
df.columns = ['Country','Region','Happiness_Score','GDP','Social_Support',
              'Life_Expectancy','Freedom','Generosity','Corruption']

print(f"Dataset: {len(df)} countries, {len(df.columns)} columns")
print(df.head())

import plotly.express as px
import plotly.graph_objects as go

# Explore the dataset before you start
print("Regions in dataset:")
print(df['Region'].value_counts())
print("\nScore range:", df['Happiness_Score'].min(), "–", df['Happiness_Score'].max())
print("\nBottom 10 countries:")
print(df.nsmallest(10, 'Happiness_Score')[['Country','Region','Happiness_Score']])


FileNotFoundError: [Errno 2] No such file or directory: '../data/world_happiness_2023.csv'

## Task 1 — Regional Comparison Bar Chart

In [1]:
# Task 1: Regional comparison bar chart
region_avg = (df.groupby('Region')['Happiness_Score']
              .mean()
              .reset_index()
              .sort_values('Happiness_Score'))  # ascending → happiest ends up on top
print(region_avg)

best, worst = region_avg.iloc[-1], region_avg.iloc[0]
gap = best['Happiness_Score'] - worst['Happiness_Score']

colors = ['#1f77b4' if r == best['Region'] else '#c7ccd1' for r in region_avg['Region']]

fig = go.Figure(go.Bar(
    x=region_avg['Happiness_Score'],
    y=region_avg['Region'],
    orientation='h',
    marker_color=colors,
    text=region_avg['Happiness_Score'].round(2),
    textposition='outside',
))
fig.update_layout(
    title=f"{best['Region']} is the happiest region, {gap:.1f} points ahead of {worst['Region']}",
    xaxis=dict(title='Average happiness score (0–10)', range=[0, 10], showgrid=True, gridcolor='#eee'),
    yaxis=dict(title=''),
    plot_bgcolor='white',
    font=dict(family='Arial', size=13),
    height=500, margin=dict(l=20, r=40, t=80, b=40),
)
fig.show()


NameError: name 'df' is not defined

## Task 2 — Bottom vs. Top: A Contrast Story

In [ ]:
# Task 2: Top 8 vs. Bottom 8 contrast
top8 = df.nlargest(8, 'Happiness_Score').copy()
top8['Group'] = 'Top 8'
bottom8 = df.nsmallest(8, 'Happiness_Score').copy()
bottom8['Group'] = 'Bottom 8'

combined = pd.concat([bottom8, top8]).sort_values('Happiness_Score')
global_avg = df['Happiness_Score'].mean()
print(f"Global average: {global_avg:.2f}")

color_map = {'Top 8': '#1f77b4', 'Bottom 8': '#d62728'}

fig = go.Figure(go.Bar(
    x=combined['Happiness_Score'],
    y=combined['Country'],
    orientation='h',
    marker_color=combined['Group'].map(color_map),
    text=combined['Happiness_Score'].round(2),
    textposition='outside',
))

# Global average reference line
fig.add_vline(x=global_avg, line_dash='dash', line_color='#555',
              annotation_text=f'Global avg {global_avg:.2f}', annotation_position='top')

# Separator between the two groups
fig.add_hline(y=7.5, line_color='#999', line_width=1)

gap = top8['Happiness_Score'].mean() - bottom8['Happiness_Score'].mean()
fig.add_annotation(x=top8['Happiness_Score'].min() - 1.5, y=7.5,
                   text=f'Gap: {gap:.1f} pts (avg)', showarrow=False,
                   font=dict(size=14, color='#333'), bgcolor='white')

fig.update_layout(
    title=f"The happiest countries score {gap:.1f} points higher than the least happy",
    xaxis=dict(title='Happiness score (0–10)', range=[0, 10], gridcolor='#eee'),
    yaxis=dict(title=''),
    plot_bgcolor='white', showlegend=False,
    font=dict(family='Arial', size=13),
    height=700, margin=dict(l=20, r=40, t=80, b=40),
)
fig.show()


## Task 3 (stretch) — Grouped bar chart

In [ ]:
# Stretch goal — grouped bar chart
regions = ['Western Europe', 'Latin America', 'East Asia', 'Sub-Saharan Africa', 'South Asia']
sub = (df[df['Region'].isin(regions)]
       .groupby('Region')[['GDP', 'Freedom']].mean().reindex(regions).reset_index())
print(sub)  # if empty/NaN, check exact names with df['Region'].unique()

fig = go.Figure()
fig.add_bar(x=sub['Region'], y=sub['GDP'], name='GDP per capita', marker_color='#1f77b4')
fig.add_bar(x=sub['Region'], y=sub['Freedom'], name='Freedom', marker_color='#ff7f0e')
top = sub.loc[sub['GDP'].idxmax(), 'Region']
fig.update_layout(
    barmode='group',
    title=f'{top} leads on GDP, but freedom scores are far more evenly spread across regions',
    yaxis=dict(title='Average score', rangemode='tozero', gridcolor='#eee'),
    plot_bgcolor='white', font=dict(family='Arial', size=13),
    legend=dict(orientation='h', y=1.1), height=500,
)
fig.show()
